# Chapter 8 Assignment — Semantic Search and RAG

**Book:** Hands-On Large Language Models (O'Reilly)  
**Chapter:** 8 — Semantic Search and Retrieval-Augmented Generation  
**Style:** You write all the code. Every exercise has a stub — fill it in.

---

## Table of Contents

| Part | Topic | Exercises |
|------|-------|-----------|
| Part 1 | Dense Retrieval | Ex 1.1 – 1.6 |
| Part 2 | Keyword Search: BM25 | Ex 2.1 – 2.2 |
| Part 3 | Reranking with a Cross-Encoder | Ex 3.1 – 3.4 |
| Part 4 | Retrieval Evaluation Metrics | Ex 4.1 – 4.4 |
| Part 5 | Basic RAG with Ollama | Ex 5.1 – 5.3 |
| Part 6 | LangChain RAG Pipeline | Ex 6.1 – 6.4 |
| Part 7 | Advanced RAG | Ex 7.1 – 7.2 |
| Part 8 | Mini Projects | A, B, C |

---

## What You Will Build

By the end of this notebook you will have a working, end-to-end RAG system that runs **100% locally** — no paid API, no internet required after the first model download. The system will:

1. Embed a text corpus and retrieve relevant chunks by meaning (dense retrieval)
2. Compare that with keyword search (BM25)
3. Rerank candidates with a cross-encoder for higher precision
4. Build an explicit **hybrid BM25 → reranker** pipeline
5. Evaluate retrieval quality with Precision@k, AP, and MAP
6. Feed retrieved context to a local LLM (Ollama) to answer questions with grounding
7. Build the same pipeline with **LangChain** to see how a production framework abstracts it
8. Rewrite queries and decompose complex questions (advanced RAG)

---

## Setup

Install dependencies once, then restart the kernel:

```bash
pip install sentence-transformers faiss-cpu rank_bm25 requests
pip install langchain langchain-community langchain-huggingface  # for Part 6
```

You also need **Ollama** running locally. Install from https://ollama.com, then pull a model:

```bash
ollama pull phi3:mini          # fast, runs on CPU (3.8B)
# OR — if you have a GPU server via SSH tunnel:
# ollama pull llama3.3:70b
```

To use your remote GPU server, open an SSH tunnel before running this notebook:
```bash
ssh -L 11434:localhost:11434 your-server
```
Then set `OLLAMA_BASE_URL` to `"http://localhost:11434"` and `OLLAMA_MODEL` to `"llama3.3:70b"` in the config cell below.

In [ ]:
# ============================================================
# CONFIG — change these two variables only
# ============================================================

OLLAMA_BASE_URL = "http://localhost:11434"   # local Ollama
OLLAMA_MODEL    = "phi3:mini"                # swap to "llama3.3:70b" for SSH tunnel

In [ ]:
# ============================================================
# IMPORTS AND CORPUS
# ============================================================

import re
import numpy as np
import requests
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# ---- Corpus: Interstellar Wikipedia summary ----
# This is the text archive we will search throughout the notebook.
INTERSTELLAR_TEXT = """
Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.
The screenplay was written by Jonathan Nolan and Christopher Nolan, based on a story developed by Jonathan Nolan.
The film stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, and Michael Caine.
It was produced by Paramount Pictures and Warner Bros. Pictures.
The story follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for humanity.
Theoretical physicist Kip Thorne, who won the 2017 Nobel Prize in Physics, served as executive producer and scientific consultant.
Thorne ensured that the depictions of relativity, wormholes, and black holes were as accurate as possible given the narrative constraints.
The film portrays the gravitational time dilation effect predicted by Einstein's general theory of relativity.
On the water planet Miller, one hour equals seven years on Earth due to proximity to the massive black hole Gargantua.
Gargantua is a fictional supermassive rotating black hole with a mass 100 million times that of the Sun.
The visual effects team worked with Thorne to produce scientifically accurate imagery of the black hole.
The result was the first physically accurate simulation of a black hole ever created for a film.
Composer Hans Zimmer created the film's score, using a church organ as the central instrument.
The score is meant to evoke themes of space, time, and the love between parents and children.
Interstellar received positive reviews from critics, who praised its ambition, visual effects, and performances.
The film grossed over 701 million dollars worldwide against a production budget of 165 million dollars.
It won the Academy Award for Best Visual Effects at the 87th Academy Awards.
Cooper, played by McConaughey, is a former NASA pilot and engineer turned farmer.
He discovers a secret NASA facility led by Professor Brand, played by Michael Caine.
Brand reveals that Earth is dying due to a blight that destroys crops and depletes oxygen.
NASA has identified a wormhole near Saturn that leads to a distant galaxy with potentially habitable planets.
The mission is to find a new planet suitable for human colonisation.
Dr. Amelia Brand, played by Anne Hathaway, is a key member of the crew.
The crew visits three candidate planets: Miller, Mann, and Edmunds.
Planet Mann is named after Dr. Mann, played by Matt Damon, who sent a deceptive distress signal.
Dr. Mann falsified data to ensure a rescue mission would come and save him.
Cooper sacrifices himself by falling into Gargantua to give Brand's ship enough thrust to reach Edmunds.
Inside Gargantua, Cooper enters a tesseract constructed by future humans who exist in five dimensions.
The tesseract allows Cooper to communicate across time by manipulating gravitational waves.
He sends the quantum data needed to solve the gravity equation back to his daughter Murph.
Murph uses the equation to enable humanity to leave Earth and populate space stations.
Cooper is eventually ejected from the tesseract and recovered near Saturn.
He wakes up on a space station where an elderly Murph is waiting for him.
The film ends with Cooper setting out to find Brand on Edmunds planet.
Interstellar premiered at the TCL Chinese Theatre in Hollywood on October 26, 2014.
It was released in 70mm film and IMAX formats as well as standard formats.
The IMAX sequences were filmed using large-format cameras for maximum visual impact.
"""

print("Corpus loaded.")
print(f"Total characters: {len(INTERSTELLAR_TEXT)}")

---

## Part 1 — Dense Retrieval

Dense retrieval turns search into geometry. You embed your documents and your query into the same vector space, then find the documents whose vectors are closest to the query vector. This works even when the query uses different words than the document — because embeddings capture *meaning*, not just letters.

The pipeline has four steps:
1. Split the corpus into sentences (your search units)
2. Embed every sentence with a pre-trained model
3. Store the embeddings in a FAISS index for fast search
4. At query time: embed the query, search the index, return top-k sentences

**Reference:** notes §2a–2c

### Exercise 1.1 — Split corpus into sentences

Split `INTERSTELLAR_TEXT` into a list of clean sentences. Strip whitespace and remove empty strings.

**Hint:** Use `re.split` or `.split('\n')` — the corpus above has one sentence per line, so splitting on newlines works well. Strip each sentence and filter out blanks.

**Expected output:**
```
Number of sentences: ~42
First sentence: 'Interstellar is a 2014 epic science fiction film...'
```

In [ ]:
def split_into_sentences(text: str) -> list[str]:
    """Split corpus text into a list of non-empty sentences."""
    # YOUR CODE HERE
    pass


sentences = split_into_sentences(INTERSTELLAR_TEXT)
print(f"Number of sentences: {len(sentences)}")
print(f"First sentence: {sentences[0]!r}")
print(f"Last sentence:  {sentences[-1]!r}")

### Exercise 1.2 — Embed with a Sentence Transformer

Load the model `'BAAI/bge-small-en-v1.5'` using `SentenceTransformer` and embed all sentences.

**Why this model?** It is near the top of the MTEB retrieval leaderboard and is small enough to run on CPU in seconds.

**Hint:** `model.encode(sentences)` returns a numpy array of shape `(num_sentences, embedding_dim)`. Convert to `float32` — FAISS requires it.

**Expected output:**
```
Embedding shape: (N, 384)   # N = number of sentences in your corpus
dtype: float32
```

In [ ]:
# Load the embedding model
# YOUR CODE HERE
embed_model = None

# Embed all sentences — shape should be (num_sentences, embedding_dim)
# YOUR CODE HERE
sentence_embeddings = None

print(f"Embedding shape: {sentence_embeddings.shape}")
print(f"dtype: {sentence_embeddings.dtype}")

### Exercise 1.3 — Build a FAISS index

Create a `faiss.IndexFlatL2` index, then add your sentence embeddings to it.

**Why IndexFlatL2?** It computes exact L2 (Euclidean) distance between the query vector and every stored vector. Perfect for small corpora — no approximation needed.

**Hint:**
- `faiss.IndexFlatL2(embedding_dim)` creates the index
- `index.add(embeddings)` adds vectors — takes a `float32` numpy array
- After adding, `index.ntotal` tells you how many vectors are stored

**Expected output:**
```
FAISS index built. Vectors stored: 42
```

In [ ]:
# YOUR CODE HERE
index = None

print(f"FAISS index built. Vectors stored: {index.ntotal}")

### Exercise 1.4 — Write a `dense_search` function

Write a function that takes a query string and returns the top-k matching sentences.

**Steps inside the function:**
1. Embed the query with `embed_model.encode([query])` — note the list wrapper
2. Cast to `float32`
3. Call `index.search(query_vec, k)` — returns `(distances, indices)`
4. Use the returned indices to look up the original sentences
5. Return a list of `(distance, sentence)` tuples

**Hint:** `index.search` returns arrays of shape `(1, k)` — use `[0]` to get the first (and only) row.

In [ ]:
def dense_search(query: str, k: int = 3) -> list[tuple[float, str]]:
    """Search the FAISS index and return top-k (distance, sentence) tuples.
    
    Lower distance = more similar (L2 distance, not similarity score).
    """
    # YOUR CODE HERE
    pass

### Exercise 1.5 — Test dense search

Run your search function with the query `"how precise was the science in this film"`.

**Expected result:** The top results should mention Kip Thorne, the Nobel Prize, scientific accuracy, and the black hole simulation — even though the query uses the word "precise" and the documents use "accurate", "scientifically", "Thorne".

This is the key insight: dense retrieval finds *meaning*, not *exact words*.

In [ ]:
QUERY = "how precise was the science in this film"

results = dense_search(QUERY, k=3)

print(f"Query: {QUERY!r}")
print("=" * 60)
for rank, (dist, sentence) in enumerate(results, start=1):
    print(f"Rank {rank} | Distance: {dist:.4f}")
    print(f"  {sentence}")
    print()

### Exercise 1.6 — When Dense Retrieval Fails (The Always-Returns Problem)

Dense retrieval has a critical failure mode: it **always returns k results**, even when the query is completely outside the corpus. FAISS measures geometric distance in embedding space — it can only find the "least far away" vectors. It has no concept of "nothing relevant exists here."

Run a query about something entirely absent from the Interstellar corpus and observe what gets returned.

**What you will see:** Results look superficially plausible (sentences about space, large numbers) but are semantically wrong. Crucially, the **L2 distances will be much higher** than your in-corpus query from Ex 1.5 — that gap is the threshold signal.

**The production fix:** Add a `max_distance` threshold to `dense_search`. Any result with `distance > threshold` is filtered out. If nothing survives, return an empty list — the honest answer.

**Tasks:**
1. Run `dense_search("What is the mass of the moon?", k=3)` and print results with distances
2. Compare average distances with your in-corpus query — observe the gap
3. Go back to your `dense_search` (Ex 1.4) and add an optional `max_distance: float = None` parameter
4. Pick a threshold from the distance comparison, then verify the OOC query returns 0 results

In [ ]:
# ---- 1. Out-of-corpus query ----
OOC_QUERY = "What is the mass of the moon?"

ooc_results = dense_search(OOC_QUERY, k=3)
ic_results  = dense_search(QUERY, k=3)       # QUERY defined in Ex 1.5

print(f"OUT-OF-CORPUS: {OOC_QUERY!r}")
print("-" * 70)
for rank, (dist, sent) in enumerate(ooc_results, 1):
    print(f"  Rank {rank} | Distance: {dist:.1f}  \u2192  {sent[:70]}")

print()
print(f"IN-CORPUS: {QUERY!r}")
print("-" * 70)
for rank, (dist, sent) in enumerate(ic_results, 1):
    print(f"  Rank {rank} | Distance: {dist:.1f}  \u2192  {sent[:70]}")

# ---- 2. Compare average distances ----
print()
avg_ooc = sum(d for d, _ in ooc_results) / len(ooc_results)
avg_ic  = sum(d for d, _ in ic_results)  / len(ic_results)
print(f"Avg distance \u2014 out-of-corpus : {avg_ooc:.1f}")
print(f"Avg distance \u2014 in-corpus     : {avg_ic:.1f}")
print("Notice: OOC distances are much larger. That gap is your threshold signal.")

# ---- 3 & 4. Add max_distance to dense_search ----
# TODO: Go back to your dense_search function (Ex 1.4) and add an optional parameter:
#
#   def dense_search(query: str, k: int = 3, max_distance: float = None):
#       ... (existing implementation) ...
#       if max_distance is not None:
#           results = [(d, s) for d, s in results if d <= max_distance]
#       return results
#
# Then uncomment and test:
# threshold = ???    # pick based on the distance gap above
# safe = dense_search(OOC_QUERY, k=3, max_distance=threshold)
# print(f"With threshold={threshold}: {len(safe)} result(s) \u2014 expected 0")

---

## Part 2 — Keyword Search: BM25

BM25 is the classic keyword search algorithm — a smarter version of word counting. It asks: does this document contain the query words? How rare are those words across the whole corpus? It has no idea that "precise" and "accurate" mean similar things — it only sees characters.

We will run the same query through BM25 and compare side by side with dense retrieval.

**Reference:** notes §2d — why BM25 fails the science query

### Exercise 2.1 — Build a BM25 index

Build a BM25 index from the `sentences` list.

**How BM25Okapi expects input:** It takes a list of *tokenized* documents — each document is a list of lowercase words, not a string.

**Hint:**
```python
tokenized = [sentence.lower().split() for sentence in sentences]
bm25 = BM25Okapi(tokenized)
```

In [ ]:
# YOUR CODE HERE
tokenized_corpus = None
bm25 = None

print(f"BM25 index built over {len(tokenized_corpus)} documents.")

### Exercise 2.2 — Run BM25 and compare side-by-side

Run the same query `QUERY` through BM25 and print its top-3 results next to the dense retrieval results from Part 1.

**How to query BM25:**
1. Tokenize the query: `query_tokens = QUERY.lower().split()`
2. Get scores: `scores = bm25.get_scores(query_tokens)` — returns an array of length `num_sentences`
3. Sort by score descending and take top-k indices

**Question to think about:** Which results does BM25 return? Why does it fail to find the science-accuracy sentences? What word in the query leads it astray?

**Hint (notes §2d):** The query contains the word "science" — BM25 will latch onto that word and find documents where "science" appears literally, not documents about scientific accuracy.

In [ ]:
def bm25_search(query: str, k: int = 3) -> list[tuple[float, str]]:
    """Search with BM25 and return top-k (score, sentence) tuples."""
    # YOUR CODE HERE
    pass


bm25_results = bm25_search(QUERY, k=3)
dense_results = dense_search(QUERY, k=3)

print(f"Query: {QUERY!r}")
print()
print(f"{'DENSE RETRIEVAL':<50} | BM25 KEYWORD SEARCH")
print("-" * 110)
for (dist, dense_sent), (score, bm25_sent) in zip(dense_results, bm25_results):
    # Truncate long sentences for display
    d_display = dense_sent[:48] + "..." if len(dense_sent) > 48 else dense_sent
    b_display = bm25_sent[:48] + "..." if len(bm25_sent) > 48 else bm25_sent
    print(f"{d_display:<50} | {b_display}")

---

## Part 3 — Reranking with a Cross-Encoder

Dense retrieval is fast but imprecise — it embeds query and document *separately*, so it never directly compares them. A **cross-encoder** (also called a reranker) reads the query and document *together* as one input and outputs a single relevance score. Much more accurate, but slower — you can't pre-compute document scores.

The production pattern is a two-stage pipeline:
1. **Retrieve** a large candidate set cheaply (BM25 or dense, top-10 or top-100)
2. **Rerank** that small set expensively with the cross-encoder, return top-3

**Reference:** notes §3a–3b

### Exercise 3.1 — Load the cross-encoder

Load `CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')` from Sentence Transformers.

This model was fine-tuned on the MS MARCO passage ranking dataset to produce a relevance score for (query, passage) pairs. The score is a raw logit — higher is better.

In [ ]:
# YOUR CODE HERE
reranker = None

print("Cross-encoder reranker loaded.")

### Exercise 3.2 — Write a `rerank` function

Write a function that takes a query and a list of candidate sentences, scores each (query, sentence) pair with the cross-encoder, and returns the candidates sorted by relevance score descending.

**How `CrossEncoder.predict` works:**
```python
pairs = [(query, sent) for sent in candidates]
scores = reranker.predict(pairs)   # returns array of float scores
```

Return a list of `(score, sentence)` tuples, sorted by score descending.

In [ ]:
def rerank(query: str, candidates: list[str]) -> list[tuple[float, str]]:
    """Score each (query, candidate) pair with the cross-encoder.
    
    Returns list of (score, sentence) sorted by score descending.
    Higher score = more relevant.
    """
    # YOUR CODE HERE
    pass

### Exercise 3.3 — BM25 → Reranker pipeline

Build the two-stage pipeline:
1. Retrieve top-10 candidates from BM25
2. Rerank those 10 with the cross-encoder
3. Return the top-3 from the reranker

Print the final top-3 results with their cross-encoder scores.

**Why use BM25 in stage 1 here?** Because we want to demonstrate that even with BM25's poor candidates, the reranker can still improve precision. In production you might use dense retrieval for stage 1 instead.

**Expected outcome:** The reranker should promote the science-relevant sentences to the top, even though BM25 retrieved mostly irrelevant candidates.

In [ ]:
# Stage 1: BM25 retrieves 10 candidates
# YOUR CODE HERE
bm25_candidates = None   # list of sentences (not tuples)

# Stage 2: Cross-encoder reranks those 10, return top-3
# YOUR CODE HERE
reranked = None

print(f"Query: {QUERY!r}")
print("\nBM25 → Reranker pipeline (top-3):")
print("=" * 60)
for rank, (score, sentence) in enumerate(reranked[:3], start=1):
    print(f"Rank {rank} | Score: {score:.4f}")
    print(f"  {sentence}")
    print()

### Exercise 3.4 — Explicit Hybrid Pipeline: BM25 → Cross-Encoder

Combining the two stages you built is the **standard production retrieval pattern**:

- **Stage 1 (BM25):** Fast, cheap, casts a wide net over many candidates
- **Stage 2 (Cross-encoder):** Slow, precise, re-scores only those candidates with full query-document attention

BM25 is used as the first stage (rather than dense retrieval) because it runs in microseconds with zero GPU cost — making it cheap to over-retrieve 10–100 candidates. The cross-encoder does the precision work on that smaller set.

Write `hybrid_search` that wires these two stages together, then compare it side-by-side with raw BM25 on the same query.

In [ ]:
def hybrid_search(query: str, top_k: int = 3, num_candidates: int = 10) -> list:
    """Two-stage retrieval: BM25 first stage followed by cross-encoder reranker.

    Args:
        query:          The search query string.
        top_k:          Final number of results to return after reranking.
        num_candidates: Number of BM25 candidates to retrieve in stage 1.
    Returns:
        List of (cross_encoder_score, sentence) tuples, sorted by score descending.
    """
    # Stage 1: BM25 retrieves a broad candidate set (cheap, fast)
    # YOUR CODE HERE
    bm25_hits = None   # call bm25_search(query, k=num_candidates)

    # Extract just the sentence strings for the reranker
    # YOUR CODE HERE
    candidates = None  # [sent for _, sent in bm25_hits]

    # Stage 2: Cross-encoder reranks only the candidates (slow, precise)
    # YOUR CODE HERE
    reranked = None    # call rerank(query, candidates)

    return reranked[:top_k]


# ---- Compare raw BM25 vs hybrid ----
bm25_top3   = bm25_search(QUERY, k=3)
hybrid_top3 = hybrid_search(QUERY, top_k=3, num_candidates=10)

print(f"Query: {QUERY!r}\n")
print(f"{'BM25 (raw score)':<55}  |  HYBRID (BM25 \u2192 Cross-Encoder score)")
print("-" * 115)
for (bscore, bsent), (hscore, hsent) in zip(bm25_top3, hybrid_top3):
    b = bsent[:52] + "..." if len(bsent) > 52 else bsent
    h = hsent[:52] + "..." if len(hsent) > 52 else hsent
    print(f"[{bscore:.3f}] {b:<55}  |  [{hscore:.3f}] {h}")
print()
print("Does the hybrid pipeline surface more science-relevant sentences than raw BM25?")

---

## Part 4 — Retrieval Evaluation Metrics

How do you know if your retrieval system is actually good? You need a test suite: a corpus, a set of queries, and **relevance judgments** — a human-labelled answer to "which sentences are relevant for this query?".

We will implement three metrics that measure different aspects of quality:
- **Precision@k** — of the top-k results, what fraction are relevant?
- **Average Precision (AP)** — does ranking position matter? (yes, it does)
- **Mean Average Precision (MAP)** — AP averaged across many queries

**Reference:** notes §4a–4d  
All exercises in this part are **pure Python** — no models needed.

---

### Relevance judgments for the Interstellar corpus

Below is a hand-labelled relevance dictionary. Each key is a query string. Each value is a list of sentence indices (0-based) that are considered relevant for that query.

In [ ]:
# Hand-labelled relevance judgments
# key   = query string
# value = list of sentence indices (into `sentences`) that are relevant

RELEVANCE = {
    "how precise was the science in this film": [5, 6, 7, 10, 11],
    "who directed interstellar":                [0],
    "what happens at the end of the film":      [30, 31, 32, 33, 34],
}

# Verify the indices make sense
print("Relevance check:")
for query, indices in RELEVANCE.items():
    print(f"\nQuery: {query!r}")
    for idx in indices:
        print(f"  [{idx}] {sentences[idx]}")

### Exercise 4.1 — Precision@k

**Formula:**
$$\text{Precision@k} = \frac{\text{number of relevant documents in top-k}}{k}$$

**Example (from notes §4b):**
```
Results (indices): [5, 0, 6]    Relevant indices: {5, 6, 7, 10, 11}

P@1 = 1/1 = 1.0   (index 5 is relevant)
P@2 = 1/2 = 0.5   (index 0 is not relevant)
P@3 = 2/3 = 0.67  (index 6 is relevant)
```

Write `precision_at_k`. It receives a list of retrieved sentence indices and a set of relevant indices.

In [ ]:
def precision_at_k(retrieved_ids: list[int], relevant_ids: set[int], k: int) -> float:
    """Fraction of top-k retrieved items that are relevant.
    
    Args:
        retrieved_ids: Ordered list of retrieved sentence indices (rank 1 first).
        relevant_ids:  Set of indices that are ground-truth relevant.
        k:             Cutoff rank.
    Returns:
        Precision@k as a float in [0, 1].
    """
    # YOUR CODE HERE
    pass


# Quick test
test_retrieved = [5, 0, 6, 10, 99]
test_relevant  = {5, 6, 7, 10, 11}

print("Precision@k test:")
for k in [1, 2, 3, 4, 5]:
    p = precision_at_k(test_retrieved, test_relevant, k)
    print(f"  P@{k} = {p:.4f}")

### Exercise 4.2 — Average Precision (AP)

Precision@k does not care *where* in the list the relevant documents appear. AP rewards systems that rank relevant documents higher by computing precision only at positions where a relevant document appears, then averaging.

**Formula:**
$$\text{AP} = \frac{1}{R} \sum_{k=1}^{n} \text{Precision@k} \times \mathbb{1}[\text{doc}_k \text{ is relevant}]$$

where $R$ = total number of relevant documents that *exist in the archive* (not just in the top-k).

**Dry-run (from notes §4c):**
```
retrieved = [5, 0, 6]    relevant = {5, 6, 7, 10, 11}    R = 5

k=1: doc 5 is relevant  → P@1 = 1/1 = 1.00  → count it
k=2: doc 0 not relevant → skip
k=3: doc 6 is relevant  → P@3 = 2/3 = 0.67  → count it

AP = (1.00 + 0.67) / 5 = 0.334
```

**Important:** Divide by `R` (total relevant in archive), not by how many you retrieved.

In [ ]:
def average_precision(retrieved_ids: list[int], relevant_ids: set[int]) -> float:
    """Average Precision for a single query.
    
    Args:
        retrieved_ids: Ordered list of retrieved sentence indices (rank 1 first).
        relevant_ids:  Set of all relevant indices that exist in the archive.
    Returns:
        Average Precision as a float in [0, 1].
    """
    # YOUR CODE HERE
    pass


# Quick test — matches the dry-run above
ap = average_precision([5, 0, 6], {5, 6, 7, 10, 11})
print(f"AP test (expected ~0.334): {ap:.4f}")

# Perfect ranking: relevant doc at position 1
ap_perfect = average_precision([5], {5})
print(f"AP perfect (expected 1.0): {ap_perfect:.4f}")

# Worst ranking: only relevant doc at position 3
ap_worst = average_precision([0, 99, 5], {5})
print(f"AP worst   (expected 0.33): {ap_worst:.4f}")

### Exercise 4.3 — Mean Average Precision (MAP)

MAP is just AP averaged across all queries in your test suite.

**Formula:**
$$\text{MAP} = \frac{1}{|Q|} \sum_{q \in Q} \text{AP}(q)$$

**Hint:** You need a retrieval function to produce `retrieved_ids` for each query. Use your existing `dense_search` or `bm25_search` — but you need the *indices* not the sentences. Write a small helper or modify your search to also return indices.

In [ ]:
def mean_average_precision(
    results_per_query:  dict[str, list[int]],
    relevant_per_query: dict[str, set[int]]
) -> float:
    """MAP across all queries.
    
    Args:
        results_per_query:  {query: [retrieved_sentence_indices]} in rank order.
        relevant_per_query: {query: {relevant_sentence_indices}}.
    Returns:
        MAP as a float in [0, 1].
    """
    # YOUR CODE HERE
    pass


# Quick sanity check with made-up numbers
fake_results   = {"q1": [0, 1, 2], "q2": [0, 1, 2], "q3": [0, 1, 2]}
fake_relevant  = {"q1": {0},       "q2": {0, 1},    "q3": {2}}
# AP(q1) = 1.0, AP(q2) = (1.0 + 1.0)/2 = 1.0, AP(q3) = (1/3)/1 = 0.33
# MAP = (1.0 + 1.0 + 0.33) / 3 = 0.778
map_test = mean_average_precision(fake_results, fake_relevant)
print(f"MAP sanity check (expected ~0.778): {map_test:.4f}")

### Exercise 4.4 — Compare dense vs. BM25

Now use your MAP function on real retrieval results. For each query in `RELEVANCE`:
1. Get the top-5 retrieved indices from `dense_search`
2. Get the top-5 retrieved indices from `bm25_search`
3. Compute MAP for each system
4. Print a comparison table

**Hint:** Your search functions return `(distance/score, sentence)` tuples. You need to recover the original *index* of each sentence. The simplest way is to look it up: `sentences.index(sentence)`. (For ties, this picks the first occurrence — acceptable for this exercise.)

**Question to think about:** Which system gets a higher MAP? Why?

In [ ]:
# Build retrieved_ids dicts for each system
# YOUR CODE HERE
dense_results_per_query = {}   # {query: [top-5 sentence indices]}
bm25_results_per_query  = {}   # {query: [top-5 sentence indices]}

relevant_per_query = {q: set(idxs) for q, idxs in RELEVANCE.items()}

# Compute MAP
# YOUR CODE HERE
dense_map = None
bm25_map  = None

print("Retrieval Evaluation (top-5, 3 queries)")
print("=" * 40)
print(f"Dense Retrieval MAP : {dense_map:.4f}")
print(f"BM25 Keyword   MAP  : {bm25_map:.4f}")
print()
print("Interpretation:")
if dense_map > bm25_map:
    print("Dense retrieval wins — meaning-based search handles these queries better.")
else:
    print("BM25 wins — exact keyword matching works well for these queries.")

---

## Part 5 — Basic RAG with Ollama

RAG = Retrieval-Augmented Generation. Instead of asking the LLM to answer from memory (where it might hallucinate), you first retrieve the relevant passages from your corpus and then *stuff them into the prompt* before the question. The LLM reads from the page, not from fuzzy training memory.

You will call Ollama using the `requests` library — no LangChain, no wrappers. Raw HTTP.

**Ollama API endpoint:** `POST {OLLAMA_BASE_URL}/api/generate`  
**Request body:** `{"model": model_name, "prompt": prompt_string, "stream": false}`  
**Response field:** `response.json()["response"]`

**Reference:** notes §5a–5b

### Exercise 5.1 — Write `ollama_generate`

Write a function that sends a prompt to your local Ollama instance and returns the generated text as a string.

**Hint:**
```python
url = f"{OLLAMA_BASE_URL}/api/generate"
payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}
response = requests.post(url, json=payload)
```

Set `stream: false` so you get one complete JSON response instead of a streaming token-by-token response.

In [ ]:
def ollama_generate(prompt: str) -> str:
    """Send a prompt to Ollama and return the generated text.
    
    Uses OLLAMA_BASE_URL and OLLAMA_MODEL from the config cell.
    """
    # YOUR CODE HERE
    pass


# Smoke test — if Ollama is running, this should return a short answer
test_response = ollama_generate("In one sentence: what is a black hole?")
print(f"Ollama smoke test:")
print(test_response)

### Exercise 5.2 — Build a RAG prompt

Write a function that assembles a grounded prompt from a question and a list of context chunks.

**Why context goes BEFORE the question (notes §5b — Lost in the Middle):**  
LLMs attend better to text at the start and end of the context window. Putting context before the question ensures the model processes the facts before it reads the question — reducing the chance it ignores them.

**Prompt template:**
```
Use ONLY the information below to answer the question. 
If the answer is not in the context, say "I don't know".

Context:
- {chunk_1}
- {chunk_2}
- {chunk_3}

Question: {question}

Answer:
```

In [ ]:
def build_rag_prompt(question: str, context_chunks: list[str]) -> str:
    """Assemble a grounded RAG prompt with context before the question.
    
    Args:
        question:       The user's question.
        context_chunks: List of retrieved sentences to use as context.
    Returns:
        A complete prompt string ready to send to the LLM.
    """
    # YOUR CODE HERE
    pass


# Preview what the assembled prompt looks like
sample_chunks = ["Kip Thorne served as scientific consultant.",
                 "The visual effects were scientifically accurate."]
sample_prompt = build_rag_prompt("How accurate was the science?", sample_chunks)
print("Assembled prompt:")
print("=" * 60)
print(sample_prompt)

### Exercise 5.3 — Write the full `rag` function

Combine everything into one pipeline function:
1. Retrieve the top-k sentences using `dense_search`
2. Extract just the sentence strings from the results
3. Build the grounded prompt with `build_rag_prompt`
4. Send it to Ollama with `ollama_generate`
5. Return the answer string

In [ ]:
def rag(question: str, k: int = 3) -> str:
    """Full RAG pipeline: retrieve → build grounded prompt → generate.
    
    Args:
        question: The user's question.
        k:        Number of chunks to retrieve.
    Returns:
        The LLM's grounded answer.
    """
    # YOUR CODE HERE
    pass


# ---- Test: WITH RAG ----
QUESTION = "How scientifically accurate is Interstellar and who ensured that?"

print("WITHOUT RAG (LLM answers from memory):")
print("-" * 60)
no_rag_answer = ollama_generate(QUESTION)
print(no_rag_answer)

print()
print("WITH RAG (LLM answers from retrieved context):")
print("-" * 60)
rag_answer = rag(QUESTION, k=3)
print(rag_answer)

---

## Part 6 — LangChain RAG Pipeline

Everything you built in Part 5 by hand — embedding, FAISS indexing, prompt assembly, and the Ollama HTTP call — LangChain wraps into composable **chain** objects. This part re-builds the same RAG pipeline using LangChain so you can compare the raw approach against a production framework.

**Why learn LangChain if you already built the raw version?**
- Real codebases use frameworks — understanding the abstractions lets you debug them faster
- `RetrievalQA` and `PromptTemplate` appear throughout the LLM ecosystem
- Swapping components (different LLM, different vector store) becomes a one-line change

**Extra install required (if you haven't run it yet):**
```bash
pip install langchain langchain-community langchain-huggingface
```

**Reference:** Book Chapter 8 — "Example: RAG with Local Models" section

### Exercise 6.1 — HuggingFace Embeddings + FAISS Vector Store via LangChain

LangChain wraps `sentence-transformers` via `HuggingFaceEmbeddings`. Instead of manually calling `model.encode()` and `index.add()`, you pass the embedding wrapper to `FAISS.from_texts()` and it handles everything internally.

**Note:** This builds a second, LangChain-managed FAISS index alongside the raw one from Part 1. Both index the same `sentences` list — you will see the results are equivalent.

After building the store, call `lc_db.similarity_search("how precise was the science", k=3)` and print the results. Each result is a `Document` object with `.page_content` (the text).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS as LangChainFAISS

# TODO: Create a HuggingFaceEmbeddings wrapper for 'BAAI/bge-small-en-v1.5'
# YOUR CODE HERE
lc_embeddings = None

# TODO: Build the FAISS vector store from `sentences` in one call
# Hint: LangChainFAISS.from_texts(sentences, lc_embeddings)
# YOUR CODE HERE
lc_db = None

# TODO: Verify it works — run a similarity search and print the results
# results = lc_db.similarity_search("how precise was the science", k=3)
# for doc in results:
#     print(doc.page_content)

print("LangChain FAISS vector store ready.")
print(f"Vectors stored: {lc_db.index.ntotal}")

### Exercise 6.2 — Ollama LLM Wrapper

LangChain's `Ollama` class wraps your local Ollama server as a standard LLM component. Once wrapped, you can pass it into any chain — LangChain handles the HTTP call, retries, and output parsing.

**Compare with Part 5:** In Part 5 you wrote `ollama_generate()` as a raw `requests.post()` call. Here LangChain handles that for you. The trade-off: convenience vs. full visibility into what happens under the hood. Knowing the raw version means you can always debug what LangChain is doing internally.

In [ ]:
from langchain_community.llms import Ollama

# TODO: Create an Ollama LLM wrapper pointing at your local server
# Hint: Ollama(base_url=OLLAMA_BASE_URL, model=OLLAMA_MODEL)
# YOUR CODE HERE
lc_llm = None

# TODO: Smoke test — invoke the LLM and print the answer
# answer = lc_llm.invoke("In one sentence: what is Retrieval-Augmented Generation?")
# print("Smoke test:", answer)

print("LangChain Ollama LLM wrapper ready.")

### Exercise 6.3 — PromptTemplate

LangChain's `PromptTemplate` is a reusable, named-variable template. You define it once and any chain instantiates it by passing keyword arguments. This is equivalent to your `build_rag_prompt` from Part 5, but it is a first-class object that chains can inspect and validate automatically.

The template below uses the same "context before question" structure from Part 5 — context goes first to counteract the Lost-in-the-Middle problem.

In [ ]:
from langchain.prompts import PromptTemplate

TEMPLATE = (
    "Use ONLY the information below to answer the question.\n"
    "If the answer is not in the context, say \"I don't know\".\n"
    "\n"
    "Context:\n"
    "{context}\n"
    "\n"
    "Question: {question}\n"
    "\n"
    "Answer:"
)

# TODO: Create the PromptTemplate object
# Hint: PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])
# YOUR CODE HERE
lc_prompt = None

# Preview the template
sample = lc_prompt.format(
    context="Kip Thorne served as scientific consultant and won the 2017 Nobel Prize.",
    question="Who ensured the science was accurate?"
)
print("Template variables:", lc_prompt.input_variables)
print()
print("Sample instantiation:")
print(sample)

### Exercise 6.4 — RetrievalQA Chain

`RetrievalQA` is LangChain's standard RAG chain. `chain_type='stuff'` means it "stuffs" all retrieved documents into the prompt at once — the simplest and most common approach for small corpora.

Wire together: `lc_db.as_retriever()` → `lc_prompt` → `lc_llm`, then invoke it and compare the answer to your raw Part 5 implementation on the same question. They should be similar — the difference is that LangChain is orchestrating the three steps for you.

In [ ]:
from langchain.chains import RetrievalQA

# TODO: Build the RetrievalQA chain
# Hint:
#   rag_chain = RetrievalQA.from_chain_type(
#       llm=lc_llm,
#       chain_type="stuff",
#       retriever=lc_db.as_retriever(search_kwargs={"k": 3}),
#       chain_type_kwargs={"prompt": lc_prompt},
#       verbose=True,
#   )
# YOUR CODE HERE
rag_chain = None

QUESTION = "How scientifically accurate is Interstellar and who ensured that?"

# TODO: Invoke the chain and print the answer
# result = rag_chain.invoke({"query": QUESTION})
# print("LangChain RAG answer:")
# print(result["result"])
#
# Then compare to your Part 5 answer:
# print("\nRaw Ollama RAG answer (Part 5):")
# print(rag(QUESTION, k=3))

---

## Part 7 — Advanced RAG

The basic RAG pipeline (retrieve → stuff → generate) works well when the user's question is clean and specific. Real questions are often messy, verbose, or compound. These two exercises tackle that.

**Reference:** notes §5e

### Exercise 7.1 — Query Rewriting

**The problem:** Users write conversational or verbose questions that confuse the retriever. For example:

> "Hey, I was just curious, I watched this movie last night, it was pretty cool, but like, I want to know — how accurate were the space things in it really?"

BM25 and even dense retrieval will struggle with that noise. The fix: ask the LLM to rewrite the question into a clean, specific search query *before* retrieval.

**Write `rewrite_query`:** Send the messy question to Ollama with a meta-prompt that asks it to produce only a clean search query (no explanation, no preamble). Then use the rewritten query in `dense_search`.

**Meta-prompt template:**
```
Rewrite the following question as a short, specific search query for a document retrieval system.
Return only the search query — no explanation, no preamble.

Question: {messy_question}

Search query:
```

In [ ]:
def rewrite_query(messy_question: str) -> str:
    """Use the LLM to rewrite a verbose question into a clean search query.
    
    Returns only the cleaned query string.
    """
    # YOUR CODE HERE
    pass


def rag_with_rewriting(question: str, k: int = 3) -> str:
    """RAG pipeline with query rewriting before retrieval."""
    # YOUR CODE HERE
    pass


MESSY_QUESTION = (
    "Hey I just watched this space movie last night and I remember there was this "
    "physics guy involved, and I think the black hole looked really real? "
    "Like how scientifically correct was all of that exactly?"
)

clean_query = rewrite_query(MESSY_QUESTION)
print(f"Original: {MESSY_QUESTION}")
print(f"\nRewritten: {clean_query!r}")

print("\nRAG answer with rewritten query:")
print("-" * 60)
print(rag_with_rewriting(MESSY_QUESTION, k=3))

### Exercise 7.2 — Multi-Query RAG

**The problem:** Some questions contain two sub-questions. For example:

> "Compare the scientific accuracy and the box office performance of Interstellar."

A single retrieval step will return chunks that are biased toward one sub-topic. The fix: decompose the question into sub-queries, run each independently, deduplicate the retrieved chunks, then generate one unified answer.

**Steps:**
1. Use the LLM to decompose the question into a list of 2–3 sub-queries (one per line)
2. Run `dense_search` for each sub-query
3. Collect all retrieved sentences, deduplicate (preserve insertion order)
4. Build a grounded prompt with the combined context
5. Generate the answer

**Decomposition meta-prompt:**
```
Decompose the following question into 2-3 specific sub-queries for document retrieval.
Return one sub-query per line, no numbering, no explanation.

Question: {question}

Sub-queries:
```

In [ ]:
def decompose_question(question: str) -> list[str]:
    """Use the LLM to decompose a compound question into sub-queries.
    
    Returns a list of query strings (one per sub-question).
    """
    # YOUR CODE HERE
    pass


def multi_query_rag(question: str, k_per_query: int = 3) -> str:
    """Multi-query RAG: decompose → retrieve per sub-query → deduplicate → generate.
    
    Args:
        question:     The compound question.
        k_per_query:  How many chunks to retrieve per sub-query.
    Returns:
        A unified answer grounded in all retrieved context.
    """
    # YOUR CODE HERE
    pass


COMPOUND_QUESTION = (
    "Compare the scientific accuracy and the box office performance of Interstellar."
)

sub_queries = decompose_question(COMPOUND_QUESTION)
print("Decomposed into sub-queries:")
for i, q in enumerate(sub_queries, 1):
    print(f"  {i}. {q}")

print("\nMulti-query RAG answer:")
print("-" * 60)
print(multi_query_rag(COMPOUND_QUESTION))

---

## Part 8 — Mini Projects

Choose **one** of the three projects below. Each one extends what you built in Parts 1–6 to a new dataset or evaluation task. These are open-ended — there is no single correct solution.

---

### Project A — Study Notes Q&A

**Goal:** Turn your own chapter 8 notes into a queryable knowledge base.

Load the chapter 8 notes markdown file, split it into paragraphs, embed them, build a FAISS index, and use your `rag` pipeline to answer questions about what you studied.

**Steps:**
1. Read the notes file into a string
2. Split by double newline (`\n\n`) to get paragraphs; filter out short ones (< 50 characters)
3. Embed the paragraphs and build a new FAISS index (separate from the Interstellar index)
4. Write a `notes_rag(question)` function that searches this notes index
5. Ask at least 3 questions about Chapter 8 content

**Example questions:**
- "What is the difference between a bi-encoder and a cross-encoder?"
- "Why does chunking with overlap help with boundary blindness?"
- "What is the lost in the middle problem?"

In [ ]:
# ============================================================
# PROJECT A — Study Notes Q&A
# ============================================================

NOTES_PATH = "../../notes/ch08-semantic-search-and-rag.md"  # adjust if needed

# YOUR CODE HERE
# 1. Load and chunk the notes
# 2. Embed paragraphs
# 3. Build FAISS index
# 4. Write notes_rag(question) function
# 5. Ask 3+ questions

---

### Project B — Recipe RAG

**Goal:** Build a recipe finder that retrieves recipes by ingredient or description.

Use the five recipes provided as string constants below. Embed them at the sentence level, build a FAISS index, and create a `recipe_rag(question)` function that answers ingredient and preparation questions.

**Example queries:**
- "What can I make with chickpeas and lemon?"
- "How do I make a quick pasta dish?"
- "What is a good recipe for someone who doesn't eat meat?"

In [ ]:
RECIPES = [
    """Lemon Chickpea Soup.
    Ingredients: 2 cans chickpeas, 1 lemon (juice and zest), 4 cups vegetable broth, 1 onion, 3 garlic cloves, cumin, coriander, olive oil, salt and pepper.
    Instructions: Sauté onion and garlic in olive oil until soft. Add cumin and coriander. Add chickpeas and broth. Simmer 15 minutes. Stir in lemon juice and zest. Season with salt and pepper. Serve with crusty bread.
    This is a vegetarian soup, ready in about 25 minutes.""",

    """Spaghetti Aglio e Olio.
    Ingredients: 400g spaghetti, 6 garlic cloves (thinly sliced), half cup olive oil, red chili flakes, fresh parsley, parmesan cheese, salt.
    Instructions: Cook spaghetti until al dente. While pasta cooks, slowly fry garlic in olive oil until golden. Add chili flakes. Toss pasta with garlic oil and pasta water. Finish with parsley and parmesan.
    This is a classic Italian pasta dish ready in under 20 minutes. It is vegetarian.""",

    """Chicken Tikka Masala.
    Ingredients: 700g chicken breast, 1 cup yogurt, tikka masala spice mix, 2 cans crushed tomatoes, 1 cup heavy cream, onion, garlic, ginger, butter.
    Instructions: Marinate chicken in yogurt and spices for 1 hour. Grill or bake chicken. Sauté onion, garlic, ginger in butter. Add tomatoes and simmer. Add cream. Add grilled chicken. Serve with rice or naan.
    This is a popular Indian curry dish with chicken. It takes about 1.5 hours including marination.""",

    """Black Bean Tacos.
    Ingredients: 2 cans black beans, corn tortillas, cumin, smoked paprika, lime juice, avocado, salsa, sour cream, cheddar cheese, cilantro.
    Instructions: Drain and rinse beans. Cook beans with cumin and smoked paprika for 5 minutes. Warm tortillas. Assemble tacos with beans, sliced avocado, salsa, sour cream, cheese, and cilantro. Squeeze lime on top.
    This is a quick vegetarian taco recipe ready in 15 minutes.""",

    """Salmon with Honey Garlic Glaze.
    Ingredients: 4 salmon fillets, 3 tablespoons honey, 3 garlic cloves (minced), 2 tablespoons soy sauce, 1 tablespoon butter, lemon, fresh dill.
    Instructions: Mix honey, garlic, and soy sauce. Sear salmon in butter 3 minutes per side. Pour glaze over salmon. Cook 2 more minutes until glaze caramelises. Serve with lemon and dill.
    This is a quick seafood recipe ready in 15 minutes. It pairs well with steamed vegetables or rice.""",
]

# ============================================================
# PROJECT B — Recipe RAG
# ============================================================

# YOUR CODE HERE
# 1. Split each recipe into sentences and track which recipe each sentence belongs to
# 2. Embed all sentences
# 3. Build FAISS index
# 4. Write recipe_rag(question) function that answers questions about recipes
# 5. Test with at least 3 different ingredient/type queries

---

### Project C — Evaluate Your RAG

**Goal:** Build a ground-truth test set for the Interstellar corpus and measure how well your `rag` function performs.

You will evaluate along two axes:
1. **Retrieval quality:** MAP of the dense_search results against your relevance judgments
2. **Answer faithfulness (manual):** For each question, read the RAG answer and judge whether it is grounded in the retrieved context or if it hallucinated

**Steps:**
1. Write 5 questions about the Interstellar corpus
2. For each question, manually identify which sentence indices are relevant (like `RELEVANCE` in Part 4)
3. Run `dense_search(question, k=5)` for each question and record the retrieved indices
4. Compute MAP across your 5 questions
5. Run `rag(question)` for each question and record the answers
6. For each answer: label it Faithful / Hallucinated / Partially faithful based on whether the answer is supported by the retrieved context
7. Print a summary table: question | MAP contribution | faithfulness label

In [ ]:
# ============================================================
# PROJECT C — Evaluate Your RAG
# ============================================================

# Step 1: Write your 5 questions and relevance judgments
MY_RELEVANCE = {
    # "your question here": [list_of_relevant_sentence_indices],
    # e.g.
    # "Who composed the Interstellar score?": [12, 13],
}

# YOUR CODE HERE
# 2. Run dense_search for each question
# 3. Compute MAP
# 4. Run rag() for each question
# 5. Label each answer: Faithful / Hallucinated / Partially faithful
# 6. Print summary table

---

## Checkpoint — What You Built

By completing this notebook you have implemented, from scratch:

| Component | Tool Used | Part |
|-----------|-----------|------|
| Sentence splitting | Python `str.split` | 1.1 |
| Dense embeddings | `sentence-transformers` BAAI/bge-small | 1.2 |
| Exact nearest-neighbour search | `faiss.IndexFlatL2` | 1.3 |
| Dense search function | FAISS `index.search` | 1.4 |
| Dense retrieval failure mode + distance threshold | Manual distance comparison | 1.6 |
| Keyword search | `rank_bm25.BM25Okapi` | 2.1–2.2 |
| Cross-encoder reranker | `CrossEncoder` ms-marco-MiniLM | 3.1–3.2 |
| BM25 → reranker pipeline | Two-stage | 3.3 |
| Explicit hybrid search function | BM25 + CrossEncoder combined | 3.4 |
| Precision@k | Pure Python | 4.1 |
| Average Precision | Pure Python | 4.2 |
| Mean Average Precision | Pure Python | 4.3–4.4 |
| Ollama raw API call | `requests.post` | 5.1 |
| Grounded RAG prompt | String assembly | 5.2 |
| Full RAG pipeline (raw) | retrieve → prompt → generate | 5.3 |
| LangChain embeddings + FAISS | `HuggingFaceEmbeddings` + `FAISS.from_texts` | 6.1 |
| LangChain LLM wrapper | `langchain_community.llms.Ollama` | 6.2 |
| LangChain PromptTemplate | `PromptTemplate` | 6.3 |
| Full RAG pipeline (LangChain) | `RetrievalQA` chain | 6.4 |
| Query rewriting | LLM meta-prompt | 7.1 |
| Multi-query decomposition | LLM decomposition + deduplication | 7.2 |